# Gợi ý người quen bằng MapReduce trên Google Colab

Chạy lần lượt các ô từ trên xuống. Không cần cài thêm thư viện. Tải bộ mã `friend-recommendation-mapreduce.zip` từ cuộc trò chuyện về máy trước. Notebook chạy mô phỏng MapReduce trên một máy Colab; chưa phải cụm Hadoop.

In [3]:
from google.colab import files
from pathlib import Path
import zipfile
import io

uploaded = files.upload()
archives = [name for name in uploaded if name.endswith(".zip")]
if len(archives) != 1:
    raise ValueError("Hãy tải lên đúng một tệp ZIP mã nguồn.")
base = Path("/content/mapreduce_demo")
base.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
    for entry in archive.infolist():
        target = (base / entry.filename).resolve()
        if not target.is_relative_to(base.resolve()):
            raise ValueError("ZIP chứa đường dẫn không hợp lệ.")
    archive.extractall(base)
project = base / "friend-recommendation-mapreduce"
if not (project / "run_local.py").is_file():
    raise ValueError("Không tìm thấy bộ mã nguồn đúng cấu trúc.")
print("Đã chuẩn bị:", project)

Saving friend-recommendation-mapreduce.zip to friend-recommendation-mapreduce (1).zip
Đã chuẩn bị: /content/mapreduce_demo/friend-recommendation-mapreduce


## 1. Chạy dữ liệu mẫu

Chương trình tạo bảng Map, Shuffle, điểm từng cặp và Top 2 trong thư mục results.

In [4]:
import subprocess
import sys

subprocess.run([sys.executable, "run_local.py"], cwd=project, check=True)

CompletedProcess(args=['/usr/bin/python3', 'run_local.py'], returncode=0)

## 2. Xem kết quả Top 2

In [5]:
import json
import pandas as pd
from IPython.display import display

rows = [json.loads(line) for line in (project / "results/recommendations.jsonl").read_text(encoding="utf-8").splitlines()]
display(pd.DataFrame([{
    "Người dùng": row["user"],
    "Gợi ý": ", ".join(f"{r['candidate']} ({r['score']})" for r in row["recommendations"]) or "Không có gợi ý"
} for row in rows]))

,Người dùng,Gợi ý
0,Alice,"Dan (1), Eve (1)"
1,Bob,"Eve (2), Frank (1)"
2,Carol,Dan (2)
3,Dan,"Carol (2), Alice (1)"
4,Eve,"Bob (2), Alice (1)"
5,Frank,"Bob (1), Eve (1)"


## 3. Xem dữ liệu trung gian

Đổi tên tệp để xem map_1.txt, shuffle_1.txt, pair_scores.txt, map_2.txt, shuffle_2.txt hoặc audit.json.

In [6]:
filename = "pair_scores.txt"
print((project / "results" / filename).read_text(encoding="utf-8"))

["PAIR", "Alice", "Dan"]	{"score": 1, "mutual_friends": ["Bob"]}
["PAIR", "Alice", "Eve"]	{"score": 1, "mutual_friends": ["Carol"]}
["PAIR", "Bob", "Eve"]	{"score": 2, "mutual_friends": ["Carol", "Dan"]}
["PAIR", "Bob", "Frank"]	{"score": 1, "mutual_friends": ["Dan"]}
["PAIR", "Carol", "Dan"]	{"score": 2, "mutual_friends": ["Bob", "Eve"]}
["PAIR", "Eve", "Frank"]	{"score": 1, "mutual_friends": ["Dan"]}
["USER", "Alice"]	null
["USER", "Bob"]	null
["USER", "Carol"]	null
["USER", "Dan"]	null
["USER", "Eve"]	null
["USER", "Frank"]	null



## 4. Chạy bộ kiểm thử

Bộ kiểm thử đối chiếu dữ liệu mẫu, trường hợp biên và 12 mạng ngẫu nhiên với thuật toán tham chiếu độc lập.

In [7]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=project, check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'unittest', 'discover', '-s', 'tests', '-v'], returncode=0)

## 5. Tùy chọn: chạy dữ liệu của bạn

Bỏ qua hai ô tiếp theo nếu chỉ dùng dữ liệu mẫu. Tải lên một tệp UTF-8 JSONL hoặc TXT, mỗi dòng có dạng:

```json
{"user":"Alice","friends":["Bob","Carol"]}
```

Kết quả riêng nằm trong results_custom, giữ kết quả mẫu trong results.

In [8]:
custom_upload = files.upload()
if len(custom_upload) != 1:
    raise ValueError("Hãy tải lên đúng một tệp dữ liệu.")
custom_name = next(iter(custom_upload))
custom_path = project / "data/custom.jsonl"
custom_path.write_bytes(custom_upload[custom_name])
subprocess.run([sys.executable, "run_local.py", "--input", str(custom_path),
                "--output", str(project / "results_custom"), "--top-k", "2"],
               cwd=project, check=True)

KeyboardInterrupt: 

In [10]:
custom_rows = [json.loads(line) for line in (project / "results_custom/recommendations.jsonl").read_text(encoding="utf-8").splitlines()]
display(pd.DataFrame([{
    "Người dùng": row["user"],
    "Gợi ý": ", ".join(f"{r['candidate']} ({r['score']})" for r in row["recommendations"]) or "Không có gợi ý"
} for row in custom_rows]))

FileNotFoundError: [Errno 2] No such file or directory: '/content/mapreduce_demo/friend-recommendation-mapreduce/results_custom/recommendations.jsonl'

## 6. Tải mã và kết quả về máy

Tệp tải xuống chứa mã nguồn, kết quả mẫu và kết quả tùy chọn nếu đã chạy. Lưu tệp trước khi phiên Colab kết thúc.

In [11]:
output_zip = Path("/content/mapreduce-colab-results.zip")
with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(project.rglob("*")):
        if path.is_file() and "__pycache__" not in path.parts:
            archive.write(path, path.relative_to(project.parent))
files.download(str(output_zip))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>